In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os


def load_results(test_function, kernel, strategy):
    save_dir = f"results/{test_function}/{strategy}/{kernel}"
    ys = []
    ts = []
    for filename in sorted(os.listdir(save_dir)):
        results = np.load(f"{save_dir}/{filename}")
        ys.append(results["ymin"])
        ts.append(results["time"])
    return np.array(ys), np.array(ts)


for test_function in ["goldstein", "lunar", "push", "rover"]:
    plt.figure(figsize=(12, 6))
    for i, (kernel, strategy, color) in enumerate(
        [
            ("matern52", "lhs", "#1f77b4"),
            ("matern52", "bfgs", "#ff7f0e"),
            ("squaredexponential", "lhs", "#2ca02c"),
            ("squaredexponential", "bfgs", "#d62728"),
        ]
    ):
        y, t = load_results(test_function, kernel, strategy)
        if y.size == 0:
            continue
        runs, max_acquisitions = y.shape
        x = np.arange(max_acquisitions)

        # ci on median
        qu = np.clip(0.5 + 1.96 * np.sqrt(0.25 / runs), 0, 1)
        ql = np.clip(0.5 - 1.96 * np.sqrt(0.25 / runs), 0, 1)
        ul = np.quantile(y, qu, axis=0)
        ll = np.quantile(y, ql, axis=0)

        plt.subplot(1, 2, 1)
        plt.plot(
            x,
            np.median(y, axis=0),
            label=f"{kernel} + {strategy} ({runs} runs)",
            color=color,
        )
        plt.fill_between(x, ll, ul, alpha=0.2, color=color)

        plt.subplot(1, 2, 2)
        plt.boxplot(y, positions=i, patch_artist=True, boxprops=dict(facecolor=color, alpha=0.5))

    plt.title(f"{test_function.capitalize()}")
    plt.xlabel("Number of Acquisitions")
    plt.ylabel("Minimum Function Value")
    plt.legend()
    plt.grid()
    plt.show()